In [18]:
# Topic 2: Regional Disparities in Testing Resources
'''
Goal (research question):
Do regions with fewer testing/prep resources also show lower GED exam participation and GED completion?
'''
# Data source:
#  5_test_candidate_cleaned_final.csv
# Definitions:
''' 
n_testing_centers = total number of testing centers
n_candidates = total number of candidates
n_prep_center_users = number of candidates who used a prep center
n_exam_takers = number of candidates who took at least one exam
n_completers = number of candidates who earned the GED credential 
'''

# State level Metrics：
'''
center_density = n_testing_centers / n_candidates         # center_density for each candidate per state

prep_coverage = n_prep_center_users / n_candidates        # how much percentage of candidates have used a prep center in candidate's state

enrollment_rate = n_exam_takers / n_candidates            # how much percentage of candidates have took at least one exam in candidate's state

completion_rate = n_completers / n_candidates             # how much percentage of candidates who earned the GED credential in candidate's state
''' 

# Outputs in this notebook:
#   1) Build candidate-level participation/completion indicators
#   2) Aggregate to state-level metrics
#   3) Descriptive charts + correlations
#   4) Candidate-level logistic models (enrollment & completion on state resources)

DATA_PATH = "../data/processed/l.csv"


In [19]:
import numpy as np
import pandas as pd

# For plots / modeling
import matplotlib.pyplot as plt

FOCUS = "state"  # "state" = plots by C_STATE; "region" = by Census region

# Optional modeling dependency
try:
    import statsmodels.api as sm
except Exception as e:
    sm = None
    print("statsmodels is not available; OLS will be skipped.")


def _first_non_null(series: pd.Series):
    series = series.dropna()
    return series.iloc[0] if len(series) else np.nan


def _any_non_null(series: pd.Series) -> bool:
    return series.notna().any()


# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "CANDIDATE_ID", "C_STATE", "T_STATE", "TEST_CENTER_ID",
    "RESULT_ID", "EXAM_START", "CREDENTIAL_DATE", "ENROLLMENT_STATUS",
    "PREP_CENTER", "ON_VUE"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in CSV: {missing}")

# Make sure categorical-ish columns are treated consistently
for c in ["C_STATE", "T_STATE", "TEST_CENTER_ID", "PREP_CENTER", "CREDENTIAL_DATE", "RESULT_ID", "EXAM_START"]:
    df[c] = df[c].replace({"": np.nan, "NA": np.nan, "NaN": np.nan})

print("Duplicate candidate rows:", int(df['CANDIDATE_ID'].duplicated().sum()))
print("Unique candidates:", int(df['CANDIDATE_ID'].nunique()))

Loaded shape: (25657, 60)
Columns: ['CANDIDATE_ATTEMPT_ID', 'CANDIDATE_ID', 'EXAM_SUBJECT', 'ON_VUE', 'GED_READY', 'LANGUAGE_CODE', 'RESULT_ID', 'EXAM_START', 'SCORE', 'FIRST_COMPLETE', 'FIRST_PASSED', 'MAKE_COMPLETE', 'MAKE_PASSED', 'TEST_CENTER_ID', 'T_NAME', 'T_STATE', 'T_ADDRESS', 'T_CITY', 'T_POSTAL_CODE', 'RESULT_ID_MISSING', 'SCORE_MISSING', 'EXAM_START_MISSING', 'ACCOUNT_SETUP_COMPLETE_DATE', 'C_CITY', 'C_STATE', 'C_COUNTY', 'BIRTH_YEAR', 'GENDER', 'LAST_YEAR_INCOME', 'HIGHEST_GRADE_COMPLETED', 'ZIP', 'JURIS_NAME', 'JURIS_POSTAL_CODE', 'TESTING_REASON', 'SCHOOL_INCOMPLETE_REASON', 'STUDIED_FOR_GED', 'STUDY_HELPFUL_ADULT_EDUCATION_CLASS', 'STUDY_HELPFUL_ADULT_EDUCATION_TEACHER', 'STUDY_HELPFUL_AUDIO_STUDY_MATERIALS', 'STUDY_HELPFUL_BOOKS_PRINTED_STUDY_MATERIAL', 'STUDY_HELPFUL_GED_READY', 'STUDY_HELPFUL_MATERIALS_MOBILE_APP', 'STUDY_HELPFUL_ONLINE_COURSE_VIDEO_STUDY_MATERIALS', 'STUDY_HELPFUL_OTHER', 'STUDY_HELPFUL_SOCIAL_NETWORKING_WEBSITE', 'STUDY_HELPFUL_TV_STUDY_PROGRAM', 'S

In [20]:
# ---------------------------
# Helper: map state -> Census region
# ---------------------------
census_regions = {
    "Northeast": ["CT","ME","MA","NH","NJ","NY","PA","RI","VT"],
    "Midwest": ["IL","IN","IA","KS","MI","MN","MO","NE","ND","OH","SD","WI"],
    "South": ["DE","DC","FL","GA","MD","NC","SC","VA","WV","AL","KY","MS","TN","AR","LA","OK","TX"],
    "West": ["AZ","CO","ID","MT","NV","NM","UT","WY","AK","CA","HI","OR","WA"],
}

state_to_region = {}
for reg, states in census_regions.items():
    for s in states:
        state_to_region[s] = reg

VALID_STATES = set(state_to_region.keys())


def add_census_region(data: pd.DataFrame, state_col: str, out_col: str):
    data = data.copy()
    data[out_col] = data[state_col].map(state_to_region)
    return data


# ---------------------------
# Step 1: candidate-level aggregation
# ---------------------------
cand = df.copy()

# Clean sentinel prep center
if "PREP_CENTER" in cand.columns:
    cand["PREP_CENTER"] = cand["PREP_CENTER"].replace({"UNKNOWN": np.nan})

cand["has_exam_attempt"] = (
    cand["RESULT_ID"].notna() | cand["EXAM_START"].notna() | cand["SCORE"].notna()
)

if cand["ENROLLMENT_STATUS"].dtype != float:
    cand["ENROLLMENT_STATUS"] = pd.to_numeric(cand["ENROLLMENT_STATUS"], errors="coerce")

cand["has_credential"] = cand["CREDENTIAL_DATE"].notna() | (cand["ENROLLMENT_STATUS"] == 4)
cand["has_prep_center"] = cand["PREP_CENTER"].notna()

if cand["ON_VUE"].dtype != float:
    cand["ON_VUE"] = pd.to_numeric(cand["ON_VUE"], errors="coerce")

cand["pct_online_attempts"] = cand.groupby("CANDIDATE_ID")["ON_VUE"].transform("mean")
cand = add_census_region(cand, state_col="C_STATE", out_col="census_region")

agg_rules = {
    "C_STATE": _first_non_null,
    "census_region": _first_non_null,
    "has_exam_attempt": "max",
    "has_credential": "max",
    "has_prep_center": "max",
    "pct_online_attempts": "mean",
    "TEST_CENTER_ID": lambda s: set(s.dropna().unique()),
    "T_STATE": _first_non_null,
}

cand_level = cand.groupby("CANDIDATE_ID").agg(agg_rules).reset_index()

print("Candidate-level shape:", cand_level.shape)
print(cand_level[['has_exam_attempt','has_credential','has_prep_center']].mean())

Candidate-level shape: (4691, 9)
has_exam_attempt    0.977190
has_credential      0.331699
has_prep_center     0.367725
dtype: float64


In [21]:
# ---------------------------
# Step 2: state-level resource + outcome metrics
# ---------------------------

# --- Problem 1: use supply-side center count ---
# Count distinct testing centers by T_STATE (where centers physically are),
# NOT by C_STATE of exam takers. Using T_STATE gives a true
# supply-side measure of how many centers exist in each state.
centers_supply = (
    df.groupby("T_STATE")["TEST_CENTER_ID"]
    .nunique()
    .rename("n_testing_centers")
    .reset_index()
    .rename(columns={"T_STATE": "C_STATE"})  # align key for merge with C_STATE
)

# Candidate counts per state
cand_counts_state = (
    cand_level.groupby("C_STATE")["CANDIDATE_ID"]
    .nunique()
    .rename("n_candidates")
)

# Participation/completion rates per state
outcomes_state = cand_level.groupby("C_STATE").agg(
    n_exam_takers=("has_exam_attempt", "sum"),
    n_completers=("has_credential", "sum"),
    prep_coverage=("has_prep_center", "mean"),
).reset_index()

state_metrics = (
    outcomes_state
    .merge(cand_counts_state.reset_index(), on="C_STATE", how="left")
    .merge(centers_supply, on="C_STATE", how="left")
)

# Prep-center supply: distinct PREP_CENTER IDs per candidate home state
prep_supply = (
    df[df["PREP_CENTER"].notna()]
    .groupby("C_STATE")["PREP_CENTER"]
    .nunique()
    .rename("n_prep_centers")
    .reset_index()
)
state_metrics = state_metrics.merge(prep_supply, on="C_STATE", how="left")

# Add census_region
state_to_region_df = cand_level[["C_STATE", "census_region"]].drop_duplicates()
state_metrics = state_metrics.merge(state_to_region_df, on="C_STATE", how="left")

state_metrics["n_testing_centers"] = state_metrics["n_testing_centers"].fillna(0)
state_metrics["n_prep_centers"] = state_metrics["n_prep_centers"].fillna(0)
state_metrics["prep_center_density"] = state_metrics["n_prep_centers"] / state_metrics["n_candidates"]
state_metrics["center_density"] = state_metrics["n_testing_centers"] / state_metrics["n_candidates"]
state_metrics["enrollment_rate"] = state_metrics["n_exam_takers"] / state_metrics["n_candidates"]
state_metrics["completion_rate"] = state_metrics["n_completers"] / state_metrics["n_candidates"]

print(f"state_metrics rows (all C_STATE values): {len(state_metrics)}")

# ---------------------------
# Problem 3 note: check non-real states (UNKNOWN / non-US abbreviations)
# ---------------------------
non_real = state_metrics[~state_metrics["C_STATE"].isin(VALID_STATES)]
real     = state_metrics[ state_metrics["C_STATE"].isin(VALID_STATES)]

print("\n[Note on state coverage]")
print(f"  Valid US state abbreviations : {len(real)} rows")
print(f"  Non-real / UNKNOWN states    : {len(non_real)} rows")
if len(non_real) > 0:
    pct = 100 * len(non_real) / len(state_metrics)
    print(f"  Non-real share               : {pct:.1f}% of all C_STATE rows")
    print(f"  Non-real C_STATE values      : {sorted(non_real['C_STATE'].dropna().unique().tolist())}")
    print(f"  Candidates in non-real states: {int(non_real['n_candidates'].sum())} "
          f"({100*non_real['n_candidates'].sum()/state_metrics['n_candidates'].sum():.1f}% of total)")
    print("  -> These rows are EXCLUDED from regression (kept in descriptive tables for transparency).")

# Census-region aggregation (use all rows including non-real for descriptive)
region_metrics = (
    state_metrics.groupby("census_region")
    .agg(
        n_candidates=("n_candidates", "sum"),
        n_testing_centers=("n_testing_centers", "sum"),
        enrollment_rate=("enrollment_rate", "mean"),
        completion_rate=("completion_rate", "mean"),
        prep_coverage=("prep_coverage", "mean"),
        center_density=("center_density", "mean"),
    )
    .reset_index()
)

print("\nregion_metrics (all regions including NaN):")
print(region_metrics)

state_metrics rows (all C_STATE values): 52

[Note on state coverage]
  Valid US state abbreviations : 51 rows
  Non-real / UNKNOWN states    : 1 rows
  Non-real share               : 1.9% of all C_STATE rows
  Non-real C_STATE values      : ['UNKNOWN']
  Candidates in non-real states: 47 (1.0% of total)
  -> These rows are EXCLUDED from regression (kept in descriptive tables for transparency).

region_metrics (all regions including NaN):
  census_region  n_candidates  n_testing_centers  enrollment_rate  \
0       Midwest           900              281.0         0.985141   
1     Northeast           516              151.0         0.962497   
2         South          1899              535.0         0.982030   
3          West          1329              341.0         0.968375   

   completion_rate  prep_coverage  center_density  
0         0.301240       0.333260        0.296903  
1         0.319762       0.379752        0.224931  
2         0.317472       0.279539        0.409388  
3  

In [22]:
# ---------------------------
# Step 3: Descriptive charts
# ---------------------------

if FOCUS == "state":
    plot_df = state_metrics.copy()
    x_label = "Center density (testing centers per candidate)"
    title_suffix = "by state (C_STATE)"
else:
    plot_df = region_metrics.copy()
    x_label = "Center density (testing centers per candidate)"
    title_suffix = "by Census region"

# Basic cleaning for plotting
plot_df = plot_df.replace([np.inf, -np.inf], np.nan)
plot_df = plot_df.dropna(subset=["center_density", "enrollment_rate", "completion_rate"])

print("Plot dataframe shape:", plot_df.shape)

# Scatter plots — font sizes match small figsize
_fs_t, _fs_l, _fs_k = 9, 8, 7

# Scatter: center_density vs enrollment_rate
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.scatter(plot_df["center_density"], plot_df["enrollment_rate"], alpha=0.8, s=18)
ax.set_title(f"Enrollment rate vs center density ({title_suffix})", fontsize=_fs_t)
ax.set_xlabel(x_label, fontsize=_fs_l)
ax.set_ylabel("Enrollment rate", fontsize=_fs_l)
ax.tick_params(axis="both", labelsize=_fs_k)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Scatter: center_density vs completion_rate
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.scatter(plot_df["center_density"], plot_df["completion_rate"], alpha=0.8, s=18)
ax.set_title(f"Completion rate vs center density ({title_suffix})", fontsize=_fs_t)
ax.set_xlabel(x_label, fontsize=_fs_l)
ax.set_ylabel("Completion rate", fontsize=_fs_l)
ax.tick_params(axis="both", labelsize=_fs_k)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Prep coverage vs outcomes
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.scatter(plot_df["prep_coverage"], plot_df["enrollment_rate"], alpha=0.8, s=18)
ax.set_title(f"Prep coverage vs enrollment rate ({title_suffix})", fontsize=_fs_t)
ax.set_xlabel("Prep coverage (share with prep center)", fontsize=_fs_l)
ax.set_ylabel("Enrollment rate", fontsize=_fs_l)
ax.tick_params(axis="both", labelsize=_fs_k)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.scatter(plot_df["prep_coverage"], plot_df["completion_rate"], alpha=0.8, s=18)
ax.set_title(f"Prep coverage vs completion rate ({title_suffix})", fontsize=_fs_t)
ax.set_xlabel("Prep coverage (share with prep center)", fontsize=_fs_l)
ax.set_ylabel("Completion rate", fontsize=_fs_l)
ax.tick_params(axis="both", labelsize=_fs_k)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Ranking tables
rank_cols = ["enrollment_rate", "completion_rate", "center_density", "prep_coverage"]
print("\nTop states/regions by enrollment_rate:")
print(plot_df.sort_values("enrollment_rate", ascending=False)[rank_cols].head(10))
print("\nTop states/regions by completion_rate:")
print(plot_df.sort_values("completion_rate", ascending=False)[rank_cols].head(10))


Plot dataframe shape: (52, 12)

Top states/regions by enrollment_rate:
    enrollment_rate  completion_rate  center_density  prep_coverage
0               1.0         0.333333        0.523810       0.380952
15              1.0         0.285714        0.428571       0.000000
50              1.0         0.400000        0.600000       0.100000
44              1.0         0.510638        0.617021       0.234043
43              1.0         0.750000        1.500000       0.000000
42              1.0         0.000000        0.333333       0.333333
41              1.0         0.482759        0.413793       0.517241
39              1.0         0.200000        0.300000       0.500000
30              1.0         0.000000        0.000000       0.000000
25              1.0         0.323944        0.295775       0.000000

Top states/regions by completion_rate:
    enrollment_rate  completion_rate  center_density  prep_coverage
26         1.000000         1.000000        0.000000       1.000000
21   

In [23]:
# Step 4: logit_run_info + VALID_STATES sample note; if sm=None set logit_run_info/m_enroll/m_complete to None
# ==============================================================
# Step 4 — Two candidate-level logistic models (redesigned framework)
# ==============================================================
# Research topic: Regional disparities in testing resources
#
# Model 1 (enrollment / participation)
#   Outcome: has_exam_attempt_i = 1 if RESULT_ID, EXAM_START, or SCORE is non-missing (else 0)
#   Question: Do state-level resources relate to taking ≥1 GED exam?
#
# Model 2 (completion)
#   Outcome: is_completer_i = 1 if CREDENTIAL_DATE present OR ENROLLMENT_STATUS == 4 (else 0)
#   Question: Do the same resources relate to ultimately earning the credential?
#
# ----- Statistical design (what we implement vs. what is ideal) -----
# • State-level resources (center_density, prep_coverage, prep_center_density) are constant
#   within each C_STATE. A full set of *state* fixed effects would absorb those regressors
#   (perfect collinearity). We therefore do NOT use 50 state dummies together with these resources.
#
# • Main specification (practical analogue to your Option A discussion):
#     logit P(Y=1) = β0 + β' resources + γ' X_i + δ' time_FE + η' census_region_FE
#   Standard errors clustered by C_STATE (~50 clusters).
#
# • Option B (random intercept u_j) is attractive in theory; full GLMM software is heavier in Python
#   (R lme4 / pymer4 / Bambi). Here: clustered SE + broad region FE is a standard applied compromise.
#
# • Time FE: account-setup year-quarter (τ_t), from ACCOUNT_SETUP_COMPLETE_DATE.
#
# Baseline controls X_i: age (BIRTH_YEAR vs setup year), GENDER, HIGHEST_GRADE_COMPLETED,
# LAST_YEAR_INCOME, ETHNICITY, TESTING_REASON, LANGUAGE_CODE.
#
# Omit from baseline (bad / post-treatment / mediators — extended models only):
#   GED_READY, ON_VUE, STUDIED_FOR_GED, STUDY_HELPFUL_*, STUDY_LOCATION_TEST_PREPARATION_CENTER,
#   ENROLLMENT_STATUS (except when defining is_completer), SCORE, FIRST_* / MAKE_* exam outcomes.
# ==============================================================

import warnings

warnings.filterwarnings("ignore")

if sm is None:
    print("statsmodels not available — install statsmodels to run Step 4 logistic models.")
    logit_key_results = None
    logit_run_info = None
    m_enroll = None
    m_complete = None
else:
    # --- 4.0 Modeling sample: candidates in valid US states only ---
    DEMO_COLS = [
        "BIRTH_YEAR",
        "GENDER",
        "LAST_YEAR_INCOME",
        "HIGHEST_GRADE_COMPLETED",
        "ETHNICITY",
        "TESTING_REASON",
        "LANGUAGE_CODE",
        "C_STATE",
        "ACCOUNT_SETUP_COMPLETE_DATE",
    ]

    cand_model = (
        df.sort_values("EXAM_START", na_position="last")
        .groupby("CANDIDATE_ID")[DEMO_COLS]
        .first()
        .reset_index()
    )

    cand_model = cand_model.merge(
        cand_level[
            [
                "CANDIDATE_ID",
                "has_exam_attempt",
                "has_credential",
                "census_region",
            ]
        ],
        on="CANDIDATE_ID",
        how="left",
    )

    res_cols = ["C_STATE", "center_density", "prep_coverage", "prep_center_density"]
    state_res = state_metrics[state_metrics["C_STATE"].isin(VALID_STATES)][res_cols].copy()
    cand_model = cand_model.merge(state_res, on="C_STATE", how="inner")

    cand_model.rename(
        columns={"has_exam_attempt": "has_exam_attempt_i", "has_credential": "is_completer_i"},
        inplace=True,
    )

    mod = cand_model.copy()
    print(
        f"Candidate-level modeling sample: {len(mod):,} rows, "
        f"{mod['C_STATE'].nunique()} states"
    )
    print(
        f"  P(has_exam_attempt_i=1) = {mod['has_exam_attempt_i'].mean():.3f}  |  "
        f"P(is_completer_i=1) = {mod['is_completer_i'].mean():.3f}"
    )

    # --- 4.1 Features ---
    acct_dt = pd.to_datetime(mod["ACCOUNT_SETUP_COMPLETE_DATE"], errors="coerce")
    acct_year = acct_dt.dt.year
    mod["age"] = (acct_year - pd.to_numeric(mod["BIRTH_YEAR"], errors="coerce")).clip(15, 80)

    mod["time_fe"] = acct_dt.dt.to_period("Q").astype(str)

    for c in ["center_density", "prep_coverage", "prep_center_density"]:
        mod[c] = pd.to_numeric(mod[c], errors="coerce")

    RESOURCE_COLS = ["center_density", "prep_coverage", "prep_center_density"]
    X_num = mod[RESOURCE_COLS + ["age"]].copy()

    CAT_COLS = [
        "GENDER",
        "LAST_YEAR_INCOME",
        "HIGHEST_GRADE_COMPLETED",
        "ETHNICITY",
        "TESTING_REASON",
        "LANGUAGE_CODE",
    ]
    X_cat_parts = []
    for c in CAT_COLS:
        X_cat_parts.append(
            pd.get_dummies(mod[c].astype("category"), prefix=c, drop_first=True, dtype=float)
        )
    X_cat = pd.concat(X_cat_parts, axis=1)

    region_fe = pd.get_dummies(
        mod["census_region"].astype("category"), prefix="reg", drop_first=True, dtype=float
    )
    time_fe = pd.get_dummies(
        mod["time_fe"].astype("category"), prefix="time", drop_first=True, dtype=float
    )

    X = pd.concat([X_num, X_cat, region_fe, time_fe], axis=1)
    X = sm.add_constant(X)
    X = X.apply(pd.to_numeric, errors="coerce")

    y_enroll = mod["has_exam_attempt_i"].astype(float)
    y_complete = mod["is_completer_i"].astype(float)
    clusters = mod["C_STATE"].astype(str)

    valid = (
        X.notna().all(axis=1)
        & y_enroll.notna()
        & y_complete.notna()
        & clusters.notna()
    )
    X_v = X.loc[valid].copy()
    y_e = y_enroll.loc[valid]
    y_c = y_complete.loc[valid]
    g = clusters.loc[valid]

    # --- DROP COLLINEAR COLUMNS to avoid LinAlgError ---
    var_zero = X_v.var() == 0
    if "const" in var_zero:
        var_zero["const"] = False
    drop_var = X_v.columns[var_zero].tolist()
    if drop_var:
        print(f"Dropping zero-variance columns: {drop_var}")
        X_v = X_v.drop(columns=drop_var)

    Q, R = np.linalg.qr(X_v)
    independent_cols = np.abs(np.diag(R)) > 1e-10
    dropped = X_v.columns[~independent_cols].tolist()
    if dropped:
        print(f"Dropping collinear columns: {dropped}")
        X_v = X_v.loc[:, independent_cols]

    print(f"Rows after listwise deletion: {valid.sum():,}")
    print(f"Columns used in model: {X_v.shape[1]}")

    def fit_logit(title, yendog):
        m = sm.Logit(yendog, X_v).fit(
            method="bfgs",
            maxiter=500,
            disp=False,
            cov_type="cluster",
            cov_kwds={"groups": g},
        )
        print("\n" + "=" * 72)
        print(title)
        print("=" * 72)
        show_idx = ["const"] + [c for c in RESOURCE_COLS + ["age"] if c in m.params.index]
        tab = m.summary2().tables[1]
        print(tab.reindex([i for i in show_idx if i in tab.index]))
        print(f"Pseudo R-squared (McFadden): {m.prsquared:.4f}  |  N = {int(m.nobs):,}")
        return m

    m_enroll = fit_logit(
        "MODEL 1 — logit P(has_exam_attempt_i=1) | resources + X_i + region FE + time FE | SE clustered by state",
        y_e,
    )
    m_complete = fit_logit(
        "MODEL 2 — logit P(is_completer_i=1) | same specification",
        y_c,
    )

    rows = []
    for label, m in [
        ("Model 1: enrollment (has_exam_attempt_i)", m_enroll),
        ("Model 2: completion (is_completer_i)", m_complete),
    ]:
        for v in RESOURCE_COLS + ["age"]:
            if v not in m.params.index:
                continue
            rows.append(
                {
                    "model": label,
                    "variable": v,
                    "coef": float(m.params[v]),
                    "cluster_SE": float(m.bse[v]),
                    "OR": float(np.exp(m.params[v])),
                    "pvalue": float(m.pvalues[v]),
                }
            )

    logit_key_results = pd.DataFrame(rows)

    logit_run_info = {
        "Research topic": "Regional disparities in testing resources (GED)",
        "Model 1 outcome": "has_exam_attempt_i = 1 if any of RESULT_ID, EXAM_START, SCORE is non-missing (else 0)",
        "Model 2 outcome": "is_completer_i = 1 if CREDENTIAL_DATE non-missing OR ENROLLMENT_STATUS == 4 (else 0)",
        "Sample restriction": "Candidates merged to C_STATE in VALID_STATES (50 states + DC); inner join with state_metrics",
        "N candidates (merged)": len(mod),
        "N states (in model sample)": int(mod["C_STATE"].nunique()),
        "N after listwise deletion": int(valid.sum()),
        "N predictors (columns in X after collinearity drop)": int(X_v.shape[1]),
        "P(has_exam_attempt_i=1)": round(float(mod.loc[valid, "has_exam_attempt_i"].mean()), 4),
        "P(is_completer_i=1)": round(float(mod.loc[valid, "is_completer_i"].mean()), 4),
        "Core state-level regressors": ", ".join(RESOURCE_COLS)
        + " (merged from state_metrics; testing centers counted by T_STATE supply)",
        "Individual controls (categorical dummies, drop_first)": ", ".join(CAT_COLS),
        "Age": "acct_setup_year - BIRTH_YEAR, clipped to [15, 80]",
        "Fixed effects — region": "Census region dummies (prefix reg_, drop_first)",
        "Fixed effects — time": "Account setup year-quarter dummies (prefix time_, drop_first)",
        "State FE": "Not included together with state-level resource regressors (collinearity)",
        "Standard errors": "Cluster-robust by candidate home state (C_STATE)",
        "Estimator": "statsmodels.discrete_model.Logit; optimization BFGS; maxiter=500",
        "Design matrix": "Zero-variance columns dropped; QR-based collinear columns removed before fit",
        "Excluded (baseline)": "GED_READY, ON_VUE, STUDIED_FOR_GED, study-help flags, prep study location, SCORE as regressor, pass/complete exam flags, etc.",
    }

    print(
        "\nNote: Coefficients are associational. For extended specs, add variables flagged as "
        "non-baseline in the Step 4 header (e.g. study behavior) only after the main table."
    )

# Descriptive state correlation matrix (valid US states) — used by export / Table C
state_metrics_valid = state_metrics[state_metrics["C_STATE"].isin(VALID_STATES)].copy()
corr_df = state_metrics_valid[
    ["center_density", "prep_coverage", "enrollment_rate", "completion_rate"]
].corr(method="pearson")
print("\n" + "=" * 72)
print("Pearson correlation — state_metrics_valid (descriptive; Word export Table C)")
print("=" * 72)
print(corr_df)


Candidate-level modeling sample: 4,644 rows, 51 states
  P(has_exam_attempt_i=1) = 0.977  |  P(is_completer_i=1) = 0.330
Dropping collinear columns: ['ETHNICITY_3']
Rows after listwise deletion: 4,644
Columns used in model: 46

MODEL 1 — logit P(has_exam_attempt_i=1) | resources + X_i + region FE + time FE | SE clustered by state
                        Coef.  Std.Err.         z         P>|z|    [0.025  \
const                6.004629  1.175183  5.109528  3.229638e-07  3.701314   
center_density      -0.039752  1.094215 -0.036329  9.710199e-01 -2.184374   
prep_coverage        0.094611  1.159056  0.081628  9.349429e-01 -2.177097   
prep_center_density  0.309495  2.010536  0.153936  8.776599e-01 -3.631084   
age                 -0.009778  0.009681 -1.010093  3.124507e-01 -0.028752   

                       0.975]  
const                8.307945  
center_density       2.104870  
prep_coverage        2.366319  
prep_center_density  4.250073  
age                  0.009195  
Pseudo R-squa

In [ ]:
# ---------------------------------------------------------------------------
# Export Topic 2 report to Word (.docx)
# Word export: Tables A–H; Model1–2 lead with Y=x+... formula; small scatter fonts; italic caption under figures
# Run after data cells + Step 4 (statsmodels). Requires: pip install python-docx
# ---------------------------------------------------------------------------
import os
import tempfile

import numpy as np
import pandas as pd

try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    raise ImportError("pip install python-docx")

if "FOCUS" not in globals():
    FOCUS = "state"

for _v in ["df", "cand_level", "state_metrics", "region_metrics"]:
    if _v not in globals():
        raise NameError(f"Run earlier cells first; missing `{_v}`.")

if "corr_df" not in globals():
    _smv = state_metrics[state_metrics["C_STATE"].isin(VALID_STATES)]
    corr_df = _smv[
        ["center_density", "prep_coverage", "enrollment_rate", "completion_rate"]
    ].corr(method="pearson")

import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt


def _df_to_doc_table(doc, dframe, caption=None, max_rows=200, float_decimals=4):
    if caption:
        p = doc.add_paragraph()
        r = p.add_run(str(caption))
        r.bold = True
    d = dframe.copy()
    if isinstance(d.index, pd.MultiIndex):
        d = d.reset_index()
    else:
        d = d.reset_index(drop=False)
        if len(d.columns) and d.columns[0] == "index" and "index" not in dframe.columns:
            d = d.drop(columns=["index"], errors="ignore")
    full_n = len(d)
    d = d.head(max_rows)
    cols = [str(c) for c in d.columns]
    tbl = doc.add_table(rows=len(d) + 1, cols=len(cols))
    tbl.style = "Table Grid"
    for j, c in enumerate(cols):
        tbl.rows[0].cells[j].text = c
    for i in range(len(d)):
        for j, c in enumerate(cols):
            v = d.iloc[i, j]
            if isinstance(v, (float, np.floating)):
                tbl.rows[i + 1].cells[j].text = (
                    f"{v:.{float_decimals}f}" if pd.notna(v) else ""
                )
            else:
                tbl.rows[i + 1].cells[j].text = "" if pd.isna(v) else str(v)
    if full_n > max_rows:
        doc.add_paragraph(f"(Table truncated: showing first {max_rows} of {full_n} rows.)")


def _summary_table_to_df(m):
    """statsmodels summary2() coef table -> DataFrame."""
    tab = m.summary2().tables[1].copy()
    tab = tab.reset_index()
    tab.rename(columns={"index": "Variable"}, inplace=True)
    return tab


def _scatter_png(xs, ys, title, xl, yl, path, title_fs=9, label_fs=8, tick_fs=7):
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    ax.scatter(xs, ys, alpha=0.8, s=18)
    ax.set_title(title, fontsize=title_fs)
    ax.set_xlabel(xl, fontsize=label_fs)
    ax.set_ylabel(yl, fontsize=label_fs)
    ax.tick_params(axis="both", labelsize=tick_fs)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


DOC_OUT = "../data/processed/t.docx"
doc = Document()
doc.add_heading("Topic 2: Regional Disparities in Testing Resources", level=0)

doc.add_heading("Research framework and variable definitions", level=1)
doc.add_paragraph(
    "Research question: Do regions with fewer testing and preparation resources also show lower GED exam participation and lower GED completion?"
)
doc.add_heading("Raw quantities (building blocks)", level=2)
doc.add_paragraph("Counts are derived from the cleaned exam-candidate file and rolled up to each candidate’s home state (C_STATE).")
doc.add_paragraph("n_testing_centers: number of distinct testing centers", style='List Bullet')
doc.add_paragraph("n_candidates: number of distinct candidates", style='List Bullet')
doc.add_paragraph("n_prep_center_users: candidates with a non-missing prep-center identifier", style='List Bullet')
doc.add_paragraph("n_exam_takers: candidates with at least one exam signal (RESULT_ID, EXAM_START, or SCORE non-missing)", style='List Bullet')
doc.add_paragraph("n_completers: candidates with a credential date or ENROLLMENT_STATUS equal to 4", style='List Bullet')
doc.add_heading("State-level metrics", level=2)
doc.add_paragraph("center_density = n_testing_centers / n_candidates (testing centers per candidate in the state)", style='List Bullet')
doc.add_paragraph("prep_coverage = n_prep_center_users / n_candidates (share of candidates using a prep center)", style='List Bullet')
doc.add_paragraph("enrollment_rate = n_exam_takers / n_candidates (share taking at least one GED exam)", style='List Bullet')
doc.add_paragraph("completion_rate = n_completers / n_candidates (share earning the GED credential)", style='List Bullet')
doc.add_paragraph("prep_center_density = n_prep_centers / n_candidates (distinct prep-center IDs per candidate, from prep-center supply merged at state level)", style='List Bullet')
doc.add_heading("Data construction and design choices", level=2)
doc.add_paragraph("The pipeline aggregates multiple exam rows per candidate to one row per CANDIDATE_ID, then to state-level metrics for charts and correlations.")
doc.add_paragraph("Testing-center supply is counted by the physical center state (T_STATE) and aligned to C_STATE for merging, so resource intensity reflects how many centers exist in a candidate’s home state rather than where each exam was administered.", style='List Bullet')
doc.add_paragraph("Enrollment rates are typically high across states, so scatter plots often show only a weak visual link between center density and participation.", style='List Bullet')
doc.add_paragraph("Completion rates vary more across states; scatter plots and state-level correlations suggest a somewhat clearer positive association of completion with both center density and prep coverage.", style='List Bullet')
doc.add_paragraph("All descriptive relationships are associational and do not imply causality.", style='List Bullet')
doc.add_paragraph("Rows with non-U.S. or UNKNOWN C_STATE may appear in the raw data. They are kept in descriptive tables for transparency but excluded from the main logistic models, which restrict to VALID_STATES (50 states plus DC) as in the notebook.", style='List Bullet')

doc.add_heading("Data summary", level=1)
doc.add_paragraph(
    f"Source CSV rows: {len(df):,}; unique candidates: {df['CANDIDATE_ID'].nunique():,}"
)
doc.add_paragraph(f"Candidate-level rows: {len(cand_level):,}")
doc.add_paragraph(f"State-level rows (all C_STATE): {len(state_metrics):,}")

doc.add_heading("Tables — Descriptive", level=1)
_df_to_doc_table(doc, region_metrics, caption="Table A — Census region metrics (aggregated)")
_df_to_doc_table(
    doc,
    state_metrics.sort_values("C_STATE"),
    caption="Table B — State-level metrics (all C_STATE)",
    max_rows=200,
)
_df_to_doc_table(
    doc,
    corr_df,
    caption="Table C — Pearson correlation (valid US states, state-level aggregates)",
)

# ----- Step 4: Model specification + results -----
doc.add_heading("Step 4 — Model specification (logistic regression)", level=1)

# Model 1 / 2: opening formula (key regressors vs controls)
doc.add_heading("Model 1 — Enrollment (has_exam_attempt_i)", level=2)
doc.add_paragraph("logit P(has_exam_attempt_i=1) = β₀ + β₁·center_density + β₂·prep_coverage + β₃·prep_center_density + β₄·age + Σ controls + region FE + quarter FE")
doc.add_paragraph("where controls include:", style='List Bullet')
doc.add_paragraph("GENDER", style='List Bullet')
doc.add_paragraph("LAST_YEAR_INCOME", style='List Bullet')
doc.add_paragraph("HIGHEST_GRADE_COMPLETED", style='List Bullet')
doc.add_paragraph("ETHNICITY", style='List Bullet')
doc.add_paragraph("TESTING_REASON", style='List Bullet')
doc.add_paragraph("LANGUAGE_CODE", style='List Bullet')
doc.add_paragraph("Standard errors are cluster-robust by C_STATE.")
doc.add_heading("Model 2 — Completion (is_completer_i)", level=2)
doc.add_paragraph("logit P(is_completer_i=1) = β₀ + β₁·center_density + β₂·prep_coverage + β₃·prep_center_density + β₄·age + Σ controls + region FE + quarter FE")
doc.add_paragraph("where controls include:", style='List Bullet')
doc.add_paragraph("GENDER", style='List Bullet')
doc.add_paragraph("LAST_YEAR_INCOME", style='List Bullet')
doc.add_paragraph("HIGHEST_GRADE_COMPLETED", style='List Bullet')
doc.add_paragraph("ETHNICITY", style='List Bullet')
doc.add_paragraph("TESTING_REASON", style='List Bullet')
doc.add_paragraph("LANGUAGE_CODE", style='List Bullet')
doc.add_paragraph("Standard errors are cluster-robust by C_STATE.")

_lri = globals().get("logit_run_info")
if _lri is not None and isinstance(_lri, dict):
    spec_df = pd.DataFrame(
        [{"Setting": k, "Description": str(v)} for k, v in _lri.items()]
    )
    _df_to_doc_table(doc, spec_df, caption="Table D — Model setup summary (both models share the same X)")
    doc.add_paragraph(
        "Interpretation: estimates are associational. State-level resource variables vary between states but are "
        "constant within state; therefore state fixed effects are not included together with those regressors."
    )
else:
    doc.add_paragraph(
        "[Model specification table skipped] Run Step 4 with statsmodels installed to create logit_run_info."
    )

_lkr = globals().get("logit_key_results")
if _lkr is not None and len(_lkr) > 0:
    _df_to_doc_table(
        doc,
        _lkr,
        caption="Table E — Key coefficients: resources + age (cluster SE by state; odds ratios = exp(coef))",
    )
else:
    doc.add_paragraph("[Table E] Run Step 4 to generate logit_key_results.")

# Fit overview
fit_rows = []
for name, m in [
    ("Model 1 — Enrollment (has_exam_attempt_i)", globals().get("m_enroll")),
    ("Model 2 — Completion (is_completer_i)", globals().get("m_complete")),
]:
    if m is None:
        continue
    fit_rows.append(
        {
            "Model": name,
            "N": int(m.nobs),
            "K_params": int(len(m.params)),
            "Pseudo_R2_McFadden": float(m.prsquared),
            "Log-Likelihood": float(m.llf),
            "AIC": float(m.aic),
            "BIC": float(m.bic),
            "Converged": bool((getattr(m, "mle_retvals", None) or {}).get("converged", True)),
        }
    )
if fit_rows:
    _df_to_doc_table(
        doc,
        pd.DataFrame(fit_rows),
        caption="Table F — Model fit overview",
        float_decimals=4,
    )

# Full coefficient tables (long — truncated for Word file size)
MAX_COEF_ROWS = 120
for label, m in [
    ("Model 1 — full coefficient table (enrollment)", globals().get("m_enroll")),
    ("Model 2 — full coefficient table (completion)", globals().get("m_complete")),
]:
    if m is None:
        continue
    coef_df = _summary_table_to_df(m)
    _df_to_doc_table(
        doc,
        coef_df,
        caption=f"Table — {label}",
        max_rows=MAX_COEF_ROWS,
        float_decimals=4,
    )

suffix = "by state" if FOCUS == "state" else "by Census region"
if FOCUS == "state":
    plot_df = state_metrics.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["center_density", "enrollment_rate", "completion_rate"]
    )
else:
    plot_df = region_metrics.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["center_density", "enrollment_rate", "completion_rate"]
    )

doc.add_heading("Tables — Rankings", level=1)
_df_to_doc_table(
    doc,
    plot_df.sort_values("enrollment_rate", ascending=False).head(15),
    caption=f"Table G — Top 15 by enrollment rate ({suffix})",
)
_df_to_doc_table(
    doc,
    plot_df.sort_values("completion_rate", ascending=False).head(15),
    caption=f"Table H — Top 15 by completion rate ({suffix})",
)

doc.add_heading("Figures", level=1)
x_label = "Center density (testing centers per candidate)"
td = tempfile.mkdtemp(prefix="t2fig_")
paths = []
for fname, title, xa, ya, xl, yl in [
    (
        "e.png",
        f"Enrollment vs center density ({suffix})",
        "center_density",
        "enrollment_rate",
        x_label,
        "Enrollment rate",
    ),
    (
        "c.png",
        f"Completion vs center density ({suffix})",
        "center_density",
        "completion_rate",
        x_label,
        "Completion rate",
    ),
    (
        "pe.png",
        f"Prep vs enrollment ({suffix})",
        "prep_coverage",
        "enrollment_rate",
        "Prep coverage",
        "Enrollment rate",
    ),
    (
        "pc.png",
        f"Prep vs completion ({suffix})",
        "prep_coverage",
        "completion_rate",
        "Prep coverage",
        "Completion rate",
    ),
]:
    fp = os.path.join(td, fname)
    _scatter_png(plot_df[xa].values, plot_df[ya].values, title, xl, yl, fp)
    paths.append((title, fp))
for title, fp in paths:
    # Title on figure only; caption line below avoids duplicate headings in Word
    doc.add_picture(fp, width=Inches(2.6))
    cap = doc.add_paragraph()
    cap.add_run(title).italic = True
for _, fp in paths:
    try:
        os.remove(fp)
    except OSError:
        pass
try:
    os.rmdir(td)
except OSError:
    pass

doc.save(DOC_OUT)
print("Saved:", DOC_OUT)
